# PROYECTO INTEGRADOR — VISIÓN COMPUTACIONAL CON OPENCV
**SENATI · Actividad completa documentada**

**Objetivo:** cargar una imagen, comprobar sus dimensiones, convertirla a escala de grises, suavizarla, detectar bordes, redimensionarla y rotarla; resolver retos y responder preguntas de reflexión.

**Aplicación interactiva:** [Abrir actividad web (archivo en GitHub)](../index.html) · **Repositorio:** https://github.com/Loren07-kie/Actividades-ml-y-dp

**Uso:** instalar dependencias, colocar `imagen_practica.png` en `recursos/` o subir una imagen propia y ejecutar las celdas en orden. Las evidencias se guardan en `resultados/`.

## 1. Preparación del entorno
Si utilizas Jupyter local, instala con `pip install opencv-python numpy matplotlib pillow`. En Google Colab también funcionan estas bibliotecas.

In [ ]:
from pathlib import Path
import cv2, numpy as np, matplotlib.pyplot as plt
from PIL import Image
OUT=Path("resultados"); OUT.mkdir(exist_ok=True)
print("OpenCV:",cv2.__version__)

## 2. Lectura y verificación de la imagen
Utilizamos la imagen del material de clase, de 900 × 560 píxeles. Si no está disponible, generamos una escena de ejemplo para que el cuaderno siga siendo ejecutable.

In [ ]:
ruta=Path("recursos/imagen_practica.png")
if ruta.exists():
    imagen=cv2.imread(str(ruta))
else:
    imagen=np.full((560,900,3),(236,225,208),np.uint8)
    cv2.rectangle(imagen,(100,120),(365,390),(65,130,245),-1)
    cv2.circle(imagen,(620,255),135,(45,190,90),-1)
    cv2.putText(imagen,"SENATI - OpenCV",(130,505),cv2.FONT_HERSHEY_SIMPLEX,1.3,(45,45,45),3)
    print("Aviso: imagen original no encontrada, se usa una imagen demostrativa.")
assert imagen is not None
h,w=imagen.shape[:2]
print(f"Ancho: {w} px | Alto: {h} px | Canales: {imagen.shape[2]}")
cv2.imwrite(str(OUT/"original.png"),imagen)
plt.imshow(cv2.cvtColor(imagen,cv2.COLOR_BGR2RGB));plt.axis("off");

## 3. Procesamiento y transformación
Escala de grises: reduce de tres canales a uno. Gauss 5×5: disminuye ruido. Canny 50/150: detecta contornos. Redimensionar 50% y rotar 15°.

In [ ]:
gris=cv2.cvtColor(imagen,cv2.COLOR_BGR2GRAY)
filtro=cv2.GaussianBlur(gris,(5,5),0)
bordes=cv2.Canny(filtro,50,150)
reducida=cv2.resize(imagen,(w//2,h//2),interpolation=cv2.INTER_AREA)
M=cv2.getRotationMatrix2D((w/2,h/2),15,1)
rotada=cv2.warpAffine(imagen,M,(w,h))
resultados={"grises":gris,"filtro":filtro,"bordes":bordes,"reducida":reducida,"rotada":rotada}
for nombre,img in resultados.items():
    cv2.imwrite(str(OUT/f"{nombre}.png"),img)
print("6 imágenes generadas en",OUT.resolve())

## 4. Comparación visual de las seis imágenes

In [ ]:
fig,ax=plt.subplots(2,3,figsize=(15,7))
for a,(nombre,im) in zip(ax.flat,[("Original",imagen),("Grises",gris),("Suavizado",filtro),("Canny",bordes),("Reducida",reducida),("Rotada 15°",rotada)]):
    if im.ndim==2:a.imshow(im,cmap="gray")
    else:a.imshow(cv2.cvtColor(im,cv2.COLOR_BGR2RGB))
    a.set_title(nombre);a.axis("off")
plt.tight_layout();plt.savefig(OUT/"comparacion.png",dpi=150);plt.show()

## 5. Reto adicional: rotación 30° y Canny 100–200
Con 30° la orientación cambia más y normalmente aumentan zonas de relleno. Umbrales 100/200 omiten bordes de bajo contraste.

In [ ]:
M30=cv2.getRotationMatrix2D((w/2,h/2),30,1)
rot30=cv2.warpAffine(imagen,M30,(w,h))
canny_extra=cv2.Canny(filtro,100,200)
cv2.imwrite(str(OUT/"rotada_30.png"),rot30)
cv2.imwrite(str(OUT/"canny_100_200.png"),canny_extra)
print("Bordes detectados Canny 50/150:",np.count_nonzero(bordes))
print("Bordes detectados Canny 100/200:",np.count_nonzero(canny_extra))

## 6. Retos resueltos de los módulos 1 a 6 y ampliación
1. **Python:** al elevar el mínimo a 150 cambian 15, 80, 110; 200 permanece intacto.
2. **Lectura/visualización:** el recorte alternativo de 250 filas por 350 columnas muestra otra zona de la imagen.
3. **Matrices:** el rectángulo entre filas 80–139 y columnas 600–849 mide 60 × 250 píxeles.
4. **RGB/grises:** umbral 70 muestra más píxeles blancos que umbral 200.
5. **Filtros:** Gauss 9×9 reduce más detalles; Canny 100–200 conserva los bordes de mayor contraste.
6. **Geometría:** rotación −25° gira en sentido horario y desplazamiento x=−60 mueve la escena a la izquierda.
7. **Ampliación:** en BGR, rojo (0,0,255) cambia a verde (0,255,0) sin alterar la forma.

In [ ]:
# Comprobación reproducible de retos de umbrales, filtros y transformaciones
_,u70=cv2.threshold(gris,70,255,cv2.THRESH_BINARY)
_,u200=cv2.threshold(gris,200,255,cv2.THRESH_BINARY)
g9=cv2.GaussianBlur(gris,(9,9),0)
Mneg=cv2.getRotationMatrix2D((w/2,h/2),-25,1)
rot_neg=cv2.warpAffine(imagen,Mneg,(w,h))
T=np.float32([[1,0,-60],[0,1,0]])
mov=cv2.warpAffine(imagen,T,(w,h))
for nombre,im in {"umbral_70":u70,"umbral_200":u200,"gauss_9x9":g9,"rotacion_menos25":rot_neg,"desplazamiento_menos60":mov}.items():cv2.imwrite(str(OUT/f"{nombre}.png"),im)
print("Píxeles blancos: umbral 70",np.count_nonzero(u70),"| umbral 200",np.count_nonzero(u200))

## 7. Preguntas de cierre respondidas
**1. ¿Diferencia entre filtro y bordes?** Un filtro modifica intensidades o reduce ruido; Canny encuentra transiciones fuertes y representa contornos.

**2. ¿Por qué escala de grises?** Reduce la imagen a un canal de intensidad y simplifica el análisis de bordes.

**3. ¿Por qué aparecen zonas vacías tras rotar?** Algunos píxeles destino quedan sin correspondencia; OpenCV por defecto los rellena de negro.

**4. ¿Cuál técnica fue más útil?** Canny, para identificar límites de objetos. El filtro Gaussiano previo ayuda a reducir contornos espurios.

**Conclusión:** las operaciones permiten distinguir estructura visual, reducir ruido y comparar transformaciones. Conviene registrar parámetros y visualizar antes/después.

**Evidencias:** revisar la carpeta `resultados/` generada al ejecutar el notebook.